In [5]:
import os
import sys
import tempfile
import torch

sys.path.insert(0, "../../cpp/build")
import kumpel_embedding
import multi_head_attention


def _with_loaded_weights(py_model, cpp_model):
    with tempfile.NamedTemporaryFile(delete=False) as fp:
        path = fp.name
    try:
        py_model.save_weights(path)
        cpp_model.load_weights(path)
    finally:
        if os.path.exists(path):
            os.remove(path)


SEED = 42

# Set Python seed
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.use_deterministic_algorithms(True)

# Create input
dimension = 512
dimension_head = 64
nheads = 12
query = torch.randn(2, 10, 512)
key = torch.randn(2, 10, 512)
value = torch.randn(2, 10, 512)

py_model = multi_head_attention.MultiHeadAttention(dimension, dimension, dimension, dimension_head, nheads)

cpp_model = kumpel_embedding.MultiHeadAttention(
    dimension, dimension, dimension, dimension_head, nheads
)

_with_loaded_weights(py_model, cpp_model)


# Forward
y_py = py_model(query, query, query)
y_cpp = cpp_model.forward(query, query, query, None, False)

# Compare
print(torch.allclose(y_py, y_cpp, atol=1e-6))

True


In [2]:
print(y_py)
print(y_cpp)

tensor([[[-0.0164,  0.0891, -0.1327,  ...,  0.0589, -0.0166,  0.0273],
         [-0.0871,  0.1198, -0.1208,  ...,  0.0565,  0.0184,  0.0037],
         [-0.0951,  0.1549, -0.1419,  ...,  0.0802,  0.0107, -0.0595],
         ...,
         [-0.1026,  0.1593, -0.1181,  ...,  0.0604,  0.0289,  0.0282],
         [-0.1297,  0.2062, -0.1462,  ...,  0.0136,  0.0087, -0.0328],
         [-0.0956,  0.1649, -0.1111,  ...,  0.0181,  0.0457, -0.0104]],

        [[-0.0205, -0.0210, -0.0418,  ..., -0.0007,  0.0121, -0.0078],
         [ 0.0358, -0.0549,  0.0092,  ..., -0.1191,  0.0183,  0.0735],
         [ 0.0363, -0.0154, -0.0394,  ..., -0.0087,  0.0623,  0.0483],
         ...,
         [ 0.0094, -0.0607, -0.0769,  ...,  0.0077, -0.0321,  0.0679],
         [ 0.0204, -0.0641, -0.0101,  ..., -0.0771,  0.0381,  0.0454],
         [-0.0281, -0.0349, -0.0010,  ..., -0.1271,  0.0079,  0.1139]]],
       grad_fn=<ViewBackward0>)
tensor([[[-0.0164,  0.0891, -0.1327,  ...,  0.0589, -0.0166,  0.0273],
         [-0.